AGGREGATED PLANNING MODEL - INTERVIEW CASE

In [ ]:
import math
import pandas as pd
import pulp as pl

# =========================
# Aggregate Planning (MILP) — 2 productos, 16 semanas
# Lead time (freezer) = 4
# Safety stock = 2 semanas
# Cero stockouts
# Camión debe usarse si hay pedido
# =========================

# -------------------------
# 1) SETS
# -------------------------
T = list(range(1, 17))               # semanas 1..16
K = ["P64", "P32"]                   # productos (64oz, 32oz)

# -------------------------
# 2) PARAMETERS
# -------------------------
lead_time = 4
safety_weeks = 2

truck_capacity_lbs = 32000

# Restricción que tú diste (lbs por pallet):
lbs_per_pallet = {"P64": 1600, "P32": 1800}

# Costos (pon los tuyos si el documento los trae)
truck_cost_per_week = 550           # costo por camión usado (semana t)
purchase_cost_per_pallet = {"P64": 262.0, "P32": 306.0}  # ejemplo realista (ajusta a tu caso)
handling_cost_per_pallet = 18
freezing_cost_per_pallet = 20
storage_cost_per_pallet_week = 5.0  # ejemplo (≈20/mes)

# Backorder = 0 (cero stockouts), pero igual dejamos B por estructura
bigM = 100000

# -------------------------
# 3) DEMAND (pallets/semana)
# -------------------------
# Ejemplo constante (cambia por tu demanda real si varía por semana)
# OJO: usa PALLETS por semana (no unidades).
D64 = 9   # pallets/semana (ejemplo)
D32 = 1    # pallets/semana (ejemplo)

D = {(k, t): (D64 if k == "P64" else D32) for k in K for t in T}

# -------------------------
# 4) INITIAL CONDITIONS (CLAVE para factibilidad con lead time)
# -------------------------
# Para sobrevivir semanas 1..lead_time SIN release y aun mantener safety stock,
# un I0 razonable y factible es: (lead_time + safety_weeks) * demanda semanal
I0 = {k: (lead_time + safety_weeks) * D[(k, 1)] for k in K}  # (4+2)=6 semanas
F0 = {k: 0 for k in K}
B0 = {k: 0 for k in K}

# -------------------------
# 5) MODEL
# -------------------------
m = pl.LpProblem("AP_GelPacks", pl.LpMinimize)

# -------------------------
# 6) VARIABLES
# -------------------------
# P[k,t]: pallets pedidos (entran a freezer)
P = pl.LpVariable.dicts("P", (K, T), lowBound=0, cat="Continuous")

# I[k,t]: inventario ready (al final de t)
I = pl.LpVariable.dicts("I", (K, T), lowBound=0, cat="Continuous")

# F[k,t]: WIP freezer (al final de t)
F = pl.LpVariable.dicts("F", (K, T), lowBound=0, cat="Continuous")

# B[k,t]: backlog (forzado a 0)
B = pl.LpVariable.dicts("B", (K, T), lowBound=0, cat="Continuous")

# W[t]: camiones usados en semana t (entero)
W = pl.LpVariable.dicts("W", T, lowBound=0, cat="Integer")

# A[t]: 1 si hay pedido en semana t (binaria)
A = pl.LpVariable.dicts("A", T, lowBound=0, upBound=1, cat="Binary")

# -------------------------
# 7) HELPERS
# -------------------------
def release(k, t):
    """Pallets que salen del freezer y quedan ready en semana t."""
    if t - lead_time >= 1:
        return P[k][t - lead_time]
    return 0

# -------------------------
# 8) OBJECTIVE
# -------------------------
m += (
    pl.lpSum(
        P[k][t] * (purchase_cost_per_pallet[k] + handling_cost_per_pallet + freezing_cost_per_pallet)
        for k in K for t in T
    )
    + pl.lpSum(storage_cost_per_pallet_week * (I[k][t] + F[k][t]) for k in K for t in T)
    + pl.lpSum(truck_cost_per_week * W[t] for t in T)
)

# -------------------------
# 9) CONSTRAINTS
# -------------------------

# (C1) Capacidad de camión (tu restricción exacta):
# 1600 P_t^64 + 1800 P_t^32 <= 32000 W_t  para todo t
for t in T:
    m += 1600 * P["P64"][t] + 1800 * P["P32"][t] <= 32000 * W[t], f"TruckCapacity_t{t}"

# (C2) Conectar "si pides -> usas camión"
# sum_k P[k,t] <= M * A[t]  y  W[t] >= A[t]
# Elegimos M como cantidad máxima de pallets si usaras, por ejemplo, 10 camiones (holgado)
M_pallets = 5 * math.ceil(truck_capacity_lbs / min(lbs_per_pallet.values()))
for t in T:
    m += pl.lpSum(P[k][t] for k in K) <= M_pallets * A[t], f"LinkOrder_A_upper_t{t}"
    m += W[t] >= A[t], f"OrderImpliesTruck_t{t}"

# (C3) Balance ready/backlog:
# (I - B)_t = (I - B)_{t-1} + release_t - demand_t
for k in K:
    for t in T:
        if t == 1:
            m += (I[k][t] - B[k][t]) == (I0[k] - B0[k]) + release(k, t) - D[(k, t)], f"ReadyBal_{k}_t{t}"
        else:
            m += (I[k][t] - B[k][t]) == (I[k][t-1] - B[k][t-1]) + release(k, t) - D[(k, t)], f"ReadyBal_{k}_t{t}"

# (C4) Balance WIP freezer:
# F_t = F_{t-1} + P_t - release_t
for k in K:
    for t in T:
        if t == 1:
            m += F[k][t] == F0[k] + P[k][t] - release(k, t), f"WIPBal_{k}_t{t}"
        else:
            m += F[k][t] == F[k][t-1] + P[k][t] - release(k, t), f"WIPBal_{k}_t{t}"

# (C5) Safety stock: I[k,t] >= 2 * D[k,t]
for k in K:
    for t in T:
        m += I[k][t] >= safety_weeks * D[(k, t)], f"Safety_{k}_t{t}"

# (C6) Cero stockouts: B[k,t] = 0
for k in K:
    for t in T:
        m += B[k][t] == 0, f"NoStockout_{k}_t{t}"

# -------------------------
# 10) SOLVE
# -------------------------
solver = pl.PULP_CBC_CMD(msg=True)
status = m.solve(solver)

print("\nSTATUS:", pl.LpStatus[status])
if pl.LpStatus[status] != "Optimal":
    raise RuntimeError("El modelo no encontró solución óptima. Revisa factibilidad (I0, demandas, safety, lead time).")

print("TOTAL COST:", pl.value(m.objective))

# -------------------------
# 11) RESULTS TABLE
# -------------------------
rows = []
for t in T:
    p64 = P["P64"][t].value()
    p32 = P["P32"][t].value()
    w = W[t].value()
    a = A[t].value()

    weight_used = 1600 * p64 + 1800 * p32
    cap = 32000 * w
    slack_cap = cap - weight_used

    r64 = P["P64"][t - lead_time].value() if t - lead_time >= 1 else 0.0
    r32 = P["P32"][t - lead_time].value() if t - lead_time >= 1 else 0.0

    rows.append({
        "t": t,
        "A_order": int(round(a)),
        "W_trucks": int(round(w)),
        "P64_order": p64,
        "P32_order": p32,
        "Rel64": r64,
        "Rel32": r32,
        "I64_ready": I["P64"][t].value(),
        "I32_ready": I["P32"][t].value(),
        "F64_wip": F["P64"][t].value(),
        "F32_wip": F["P32"][t].value(),
        "cap_used_lbs": weight_used,
        "cap_total_lbs": cap,
        "cap_slack_lbs": slack_cap
    })

df = pd.DataFrame(rows)

print("\n--- PLAN (16 semanas) ---")
print(df.to_string(index=False))

# -------------------------
# 12) QUICK CHECKS (evidencia de cumplimiento)
# -------------------------
print("\n--- CHECKS ---")

# Check 1: si A=1 entonces W>=1
viol = df[(df["A_order"] == 1) & (df["W_trucks"] < 1)]
print("Pedido sin camión:", "OK" if viol.empty else "FALLA")
if not viol.empty:
    print(viol[["t","A_order","W_trucks"]])

# Check 2: capacity slack no negativo
viol = df[df["cap_slack_lbs"] < -1e-6]
print("Capacidad camión:", "OK" if viol.empty else "FALLA")
if not viol.empty:
    print(viol[["t","cap_used_lbs","cap_total_lbs","cap_slack_lbs"]])

# Check 3: safety stock (I >= 2D)
# Como D es constante aquí, lo chequeamos directo
min_I64 = df["I64_ready"].min()
min_I32 = df["I32_ready"].min()
print(f"Safety 64oz: min I = {min_I64:.2f} vs req = {2*D64:.2f}  ->", "OK" if min_I64 + 1e-6 >= 2*D64 else "FALLA")
print(f"Safety 32oz: min I = {min_I32:.2f} vs req = {2*D32:.2f}  ->", "OK" if min_I32 + 1e-6 >= 2*D32 else "FALLA")

# Check 4: backlog (debe ser 0)
# (si quieres, puedes imprimir B pero aquí ya está forzado)
print("Backorders:", "OK (forzado a 0)")



STATUS: Optimal
TOTAL COST: 44942.4444784

--- PLAN (16 semanas) ---
 t  A_order  W_trucks  P64_order  P32_order  Rel64    Rel32  I64_ready  I32_ready  F64_wip  F32_wip  cap_used_lbs  cap_total_lbs  cap_slack_lbs
 1        1         1        9.0   1.666667    0.0 0.000000       45.0   5.000000      9.0 1.666667   17400.00006        32000.0    14599.99994
 2        1         1       18.0   1.777778    0.0 0.000000       36.0   4.000000     27.0 3.444444   32000.00004        32000.0       -0.00004
 3        0         0        0.0   0.000000    0.0 0.000000       27.0   3.000000     27.0 3.444444       0.00000            0.0        0.00000
 4        1         1       18.0   1.777778    0.0 0.000000       18.0   2.000000     45.0 5.222222   32000.00004        32000.0       -0.00004
 5        0         0        0.0   0.000000    9.0 1.666667       18.0   2.666667     36.0 3.555556       0.00000            0.0        0.00000
 6        1         1       18.0   1.777778   18.0 1.777778       